## Setup and Imports

In [ ]:
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# lifelines provides the Kaplan-Meier estimator and log-rank test
# KaplanMeierFitter fits and plots the survival curve
# multivariate_logrank_test tests whether multiple groups differ significantly
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test

from utils.helpers import set_style, save_figure, start_logging
set_style()
log = start_logging(project_root, '04_02_kaplan_meier')

print('All imports successful')
print(f'lifelines version: {__import__("lifelines").__version__}')


## Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'

# Load the survival dataset from Notebook 1
df_raw = pd.read_csv(processed_path / 'survival_data.csv')
df = df_raw.copy()

# Load churn configuration
config = pd.read_csv(processed_path / 'churn_config.csv').iloc[0]
CHURN_THRESHOLD = int(config['churn_threshold_days'])

print(f'Survival dataset: {len(df):,} customers')
print(f'Churned: {df["churned"].sum():,} ({df["churned"].mean()*100:.1f}%)')
print(f'Censored: {(df["churned"]==0).sum():,} ({(df["churned"]==0).mean()*100:.1f}%)')
print(f'Churn threshold: {CHURN_THRESHOLD} days')
print(f'Duration range: {df["duration"].min():.0f} to {df["duration"].max():.0f} days')

log(f'Survival dataset loaded: {len(df):,} customers')
log(f'Churn threshold: {CHURN_THRESHOLD} days')

## Overall Survival Curve


In [ ]:
# Fit the overall Kaplan-Meier curve
# T = duration (days from first purchase to churn or observation end)
# E = event observed (1 = churned, 0 = censored/still active)
kmf_overall = KaplanMeierFitter(label='All customers')
kmf_overall.fit(
    durations  = df['duration'],
    event_observed = df['churned']
)

median_survival = kmf_overall.median_survival_time_

print('Overall Kaplan-Meier survival curve fitted')
print(f'Median survival time: {median_survival:.0f} days')
print(f'  (50% of customers churn within {median_survival:.0f} days of first purchase)')
print()

# Survival probability at key time points
for days in [90, 180, 365, 500]:
    prob = kmf_overall.survival_function_at_times(days).values[0]
    print(f'  Survival at {days:>3} days: {prob:.1%} of customers still active')

log(f'Median survival time: {median_survival:.0f} days')

## Survival by RFM Segment

In [ ]:
# Fit Kaplan-Meier for each RFM segment
segments = ['Champions', 'Loyal Customers', 'At Risk', 'Lapsed']
seg_colors = {
    'Champions':      '#1F4E79',
    'Loyal Customers':'#2E75B6',
    'At Risk':        '#ED7D31',
    'Lapsed':         '#C00000'
}

kmf_segments = {}
seg_medians  = {}

print('Kaplan-Meier by RFM segment')
print(f'{"Segment":<20} {"N":>6} {"Events":>8} {"Median (days)":>15}')
print('-' * 53)

for seg in segments:
    mask = df['segment'] == seg
    sub  = df[mask]
    if len(sub) < 10:
        continue
    kmf = KaplanMeierFitter(label=seg)
    kmf.fit(sub['duration'], event_observed=sub['churned'])
    kmf_segments[seg] = kmf
    median = kmf.median_survival_time_
    seg_medians[seg]  = median
    events = sub['churned'].sum()
    print(f'{seg:<20} {len(sub):>6,} {events:>8,} {median:>15.0f}')

log('KM curves fitted for all segments')
for seg, median in seg_medians.items():
    log(f'  {seg}: median {median:.0f} days')

In [ ]:
# Log-rank test across all segments
# Tests H0: all segment survival curves are the same
# If p < 0.05 the curves are significantly different
df_seg_test = df[df['segment'].isin(segments)].dropna(subset=['segment'])

results_seg = multivariate_logrank_test(
    df_seg_test['duration'],
    df_seg_test['segment'],
    df_seg_test['churned']
)

print('Log-rank test: are segment survival curves significantly different?')
print(f'  Test statistic: {results_seg.test_statistic:.4f}')
print(f'  p-value:        {results_seg.p_value:.6f}')
if results_seg.p_value < 0.05:
    print('  Result: p < 0.05 - survival curves differ significantly across segments')
else:
    print('  Result: p >= 0.05 - no significant difference between segment curves')

log(f'Segment log-rank p-value: {results_seg.p_value:.6f}')

## Survival by Channel and Region

In [ ]:
# Kaplan-Meier by marketing channel
channels = ['direct', 'email', 'affiliate', 'social media']
channel_colors = {
    'direct':       '#1F4E79',
    'email':        '#2E75B6',
    'affiliate':    '#70AD47',
    'social media': '#ED7D31'
}

kmf_channels = {}
ch_medians   = {}

print('Kaplan-Meier by marketing channel')
print(f'{"Channel":<16} {"N":>6} {"Events":>8} {"Median (days)":>15}')
print('-' * 49)

for ch in channels:
    mask = df['primary_channel'] == ch
    sub  = df[mask]
    if len(sub) < 30:
        continue
    kmf = KaplanMeierFitter(label=ch)
    kmf.fit(sub['duration'], event_observed=sub['churned'])
    kmf_channels[ch] = kmf
    median = kmf.median_survival_time_
    ch_medians[ch]   = median
    events = sub['churned'].sum()
    print(f'{ch:<16} {len(sub):>6,} {events:>8,} {median:>15.0f}')

# Log-rank test
df_ch_test = df[df['primary_channel'].isin(channels)].dropna(subset=['primary_channel'])
results_ch = multivariate_logrank_test(
    df_ch_test['duration'],
    df_ch_test['primary_channel'],
    df_ch_test['churned']
)
print()
print(f'Log-rank test p-value: {results_ch.p_value:.6f}')
if results_ch.p_value < 0.05:
    print('Result: survival curves differ significantly across channels')
else:
    print('Result: no significant difference between channel curves')

log(f'Channel log-rank p-value: {results_ch.p_value:.6f}')
for ch, median in ch_medians.items():
    log(f'  {ch}: median {median:.0f} days')

In [ ]:
# Kaplan-Meier by region
regions = ['Americas', 'EMEA', 'APAC', 'LATAM']
region_colors = {
    'Americas': '#1F4E79',
    'EMEA':     '#2E75B6',
    'APAC':     '#70AD47',
    'LATAM':    '#ED7D31'
}

kmf_regions = {}
reg_medians = {}

print('Kaplan-Meier by region')
print(f'{"Region":<12} {"N":>6} {"Events":>8} {"Median (days)":>15}')
print('-' * 45)

for reg in regions:
    mask = df['primary_region'] == reg
    sub  = df[mask]
    if len(sub) < 30:
        continue
    kmf = KaplanMeierFitter(label=reg)
    kmf.fit(sub['duration'], event_observed=sub['churned'])
    kmf_regions[reg] = kmf
    median = kmf.median_survival_time_
    reg_medians[reg] = median
    events = sub['churned'].sum()
    print(f'{reg:<12} {len(sub):>6,} {events:>8,} {median:>15.0f}')

# Log-rank test
df_reg_test = df[df['primary_region'].isin(regions)].dropna(subset=['primary_region'])
results_reg = multivariate_logrank_test(
    df_reg_test['duration'],
    df_reg_test['primary_region'],
    df_reg_test['churned']
)
print()
print(f'Log-rank test p-value: {results_reg.p_value:.6f}')
if results_reg.p_value < 0.05:
    print('Result: survival curves differ significantly across regions')
else:
    print('Result: no significant difference between region curves')

log(f'Region log-rank p-value: {results_reg.p_value:.6f}')
for reg, median in reg_medians.items():
    log(f'  {reg}: median {median:.0f} days')

---
## Section 5 - Visualisations

In [ ]:
# Chart 1: Overall survival curve
fig, ax = plt.subplots(figsize=(12, 5))

kmf_overall.plot_survival_function(
    ax=ax, color='#2E75B6', linewidth=2,
    ci_show=True, ci_alpha=0.15
)

# Mark the median survival time
ax.axhline(0.5, color='#C00000', linestyle='--', linewidth=1,
           alpha=0.7, label='50% survival mark')
ax.axvline(median_survival, color='#C00000', linestyle='--',
           linewidth=1, alpha=0.7)
ax.annotate(f'Median: {median_survival:.0f} days',
            xy=(median_survival, 0.5),
            xytext=(median_survival + 20, 0.55),
            fontsize=10, color='#C00000')

ax.set_title('Overall customer survival curve\n'
             'Shaded area = 95% confidence interval',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Days since first purchase')
ax.set_ylabel('Probability of remaining active')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

save_figure(fig, '04_02_overall_survival_curve.png')
plt.show()

In [ ]:
# Chart 2: Survival curves by RFM segment
fig, ax = plt.subplots(figsize=(13, 6))

for seg, kmf in kmf_segments.items():
    kmf.plot_survival_function(
        ax=ax,
        color=seg_colors.get(seg, 'grey'),
        linewidth=2,
        ci_show=False
    )

ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, alpha=0.5)
ax.set_title(f'Survival curves by RFM segment\n'
             f'Log-rank test p-value: {results_seg.p_value:.4f} - '
             f'{"significant" if results_seg.p_value < 0.05 else "not significant"}',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Days since first purchase')
ax.set_ylabel('Probability of remaining active')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

save_figure(fig, '04_02_survival_by_segment.png')
plt.show()

In [ ]:
# Chart 3: Survival curves by marketing channel
fig, ax = plt.subplots(figsize=(13, 6))

for ch, kmf in kmf_channels.items():
    kmf.plot_survival_function(
        ax=ax,
        color=channel_colors.get(ch, 'grey'),
        linewidth=2,
        ci_show=False
    )

ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, alpha=0.5)
ax.set_title(f'Survival curves by marketing channel\n'
             f'Log-rank test p-value: {results_ch.p_value:.4f} - '
             f'{"significant" if results_ch.p_value < 0.05 else "not significant"}',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Days since first purchase')
ax.set_ylabel('Probability of remaining active')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

save_figure(fig, '04_02_survival_by_channel.png')
plt.show()

In [ ]:
# Chart 4: Survival curves by region
fig, ax = plt.subplots(figsize=(13, 6))

for reg, kmf in kmf_regions.items():
    kmf.plot_survival_function(
        ax=ax,
        color=region_colors.get(reg, 'grey'),
        linewidth=2,
        ci_show=False
    )

ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, alpha=0.5)
ax.set_title(f'Survival curves by region\n'
             f'Log-rank test p-value: {results_reg.p_value:.4f} - '
             f'{"significant" if results_reg.p_value < 0.05 else "not significant"}',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Days since first purchase')
ax.set_ylabel('Probability of remaining active')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

save_figure(fig, '04_02_survival_by_region.png')
plt.show()

In [ ]:
# Chart 5: Median survival time comparison across groups
# A single chart summarising the key metric from all three stratifications

all_medians = {}
for seg, m in seg_medians.items():
    all_medians[f'Seg: {seg}'] = m
for ch, m in ch_medians.items():
    all_medians[f'Ch: {ch}'] = m
for reg, m in reg_medians.items():
    all_medians[f'Reg: {reg}'] = m

medians_series = pd.Series(all_medians).sort_values(ascending=True)
overall_med    = kmf_overall.median_survival_time_

fig, ax = plt.subplots(figsize=(12, 7))

colors = []
for label in medians_series.index:
    if label.startswith('Seg'):
        colors.append('#2E75B6')
    elif label.startswith('Ch'):
        colors.append('#70AD47')
    else:
        colors.append('#ED7D31')

bars = ax.barh(medians_series.index, medians_series.values, # pyright: ignore[reportArgumentType]
               color=colors, alpha=0.85)
ax.axvline(overall_med, color='#C00000', linestyle='--',
           linewidth=1.5, label=f'Overall median: {overall_med:.0f} days')

for bar, val in zip(bars, medians_series.values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} days', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2E75B6', label='RFM Segment'),
    Patch(facecolor='#70AD47', label='Channel'),
    Patch(facecolor='#ED7D31', label='Region'),
]
ax.legend(handles=legend_elements + [
    plt.Line2D([0], [0], color='#C00000', linestyle='--',
               label=f'Overall median: {overall_med:.0f} days')
], fontsize=9)

ax.set_title('Median survival time by group\n'
             'Blue = RFM segment, Green = channel, Orange = region',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Median survival time (days)')

save_figure(fig, '04_02_median_survival_comparison.png')
plt.show()

---
## Section 6 - Findings

In [ ]:
longest_seg  = max(seg_medians, key=seg_medians.get) # pyright: ignore[reportArgumentType, reportCallIssue]
shortest_seg = min(seg_medians, key=seg_medians.get) # pyright: ignore[reportArgumentType, reportCallIssue]
longest_ch   = max(ch_medians,  key=ch_medians.get) # pyright: ignore[reportArgumentType, reportCallIssue]
shortest_ch  = min(ch_medians,  key=ch_medians.get) # pyright: ignore[reportArgumentType, reportCallIssue]
longest_reg  = max(reg_medians, key=reg_medians.get) # pyright: ignore[reportArgumentType, reportCallIssue]
shortest_reg = min(reg_medians, key=reg_medians.get) # pyright: ignore[reportArgumentType, reportCallIssue]

print('Notebook 2 Findings - Kaplan-Meier Survival Analysis')
print()
print('Overall survival')
print(f'  Median survival time: {median_survival:.0f} day')
print(f'  Survival at 90 days:  27.8% of customers still active')
print(f'  Survival at 180 days: 26.8%')
print(f'  Survival at 365 days: 18.3%')
print(f'  Survival at 500 days: 10.0%')
print()
print('Finding 1 - Median of 1 day is technically correct but not interpretable')
print('  The median survival of 1 day reflects that 72.5% of customers')
print('  made exactly one purchase. Their survival duration from first to')
print('  last purchase is zero (clipped to 1). Since more than 50% of')
print('  all customers are one-time buyers, the 50% survival threshold')
print('  is crossed immediately. The meaningful metric is the survival')
print('  probability at specific time horizons rather than the median.')
print('  At 90 days, 27.8% of customers are still active.')
print('  At 365 days, 18.3% are still active.')
print()
print('Finding 2 - Champions are the only segment with meaningful survival signal')
print(f'  Champions median survival: {seg_medians["Champions"]:.0f} days - the only segment above 1 day')
print(f'  Champions start at 53% survival probability at day 0,')
print(f'  meaning nearly half are already one-time buyers even in this segment.')
print(f'  The Champions curve declines gradually to 11% by day 750,')
print(f'  showing these customers have genuine tenure that can be modelled.')
print(f'  All other segments drop to their floor within the first few days.')
print(f'  Log-rank p-value: {results_seg.p_value:.6f} - segment differences are significant.')
print()
print('Finding 3 - Email channel customers survive longest')
print(f'  Email starts at 34% survival and maintains the flattest curve')
print(f'  throughout 750 days - the most stable channel.')
print(f'  Social media drops vertically at day 180 as all social media')
print(f'  one-time buyers cross the churn threshold simultaneously.')
print(f'  Direct channel shows the steepest ongoing decline.')
print(f'  Log-rank p-value: {results_ch.p_value:.6f} - channel differences are significant.')
print(f'  This confirms the Notebook 1 finding that email has lower churn')
print(f'  rate despite lower CLV - email customers stay engaged longer.')
print()
print('Finding 4 - LATAM shows unexpectedly flat survival curve')
print(f'  LATAM holds near 29% survival from day 200 through to day 750')
print(f'  with almost no further decline. This means LATAM customers who')
print(f'  survived the initial one-time buyer drop are extremely long-tenured.')
print(f'  Americas and EMEA show much steeper ongoing decline.')
print(f'  Region log-rank p-value: {results_reg.p_value:.6f} - significant.')
print()
print('Finding 5 - The survival curves confirm the dataset structure')
print('  Every curve shows the same pattern: a sharp drop at day 0-1')
print('  followed by a slow gradual decline. The initial drop is the')
print('  one-time buyer population being classified as churned immediately.')
print('  The slow tail is the smaller population of repeat buyers and')
print('  recently acquired customers who are still within their window.')
print('  This is an honest picture of a low-repeat gaming hardware retailer.')
print()
print('Finding 6 - Kaplan-Meier is descriptive, not causal')
print('  These curves show WHAT happened - which groups survive longer.')
print('  They do not explain WHY. The Cox proportional hazards model')
print('  in Notebook 3 will control for multiple factors simultaneously')
print('  to isolate the independent effect of each feature on churn.')

---
## Section 7 - Export

In [ ]:
# Save median survival times for use in the Excel report
survival_summary = []

survival_summary.append({
    'group_type': 'Overall',
    'group':      'All customers',
    'n':          len(df),
    'events':     int(df['churned'].sum()),
    'median_survival_days': median_survival,
    'logrank_p':  None
})

for seg, kmf in kmf_segments.items():
    sub = df[df['segment'] == seg]
    survival_summary.append({
        'group_type': 'Segment',
        'group':      seg,
        'n':          len(sub),
        'events':     int(sub['churned'].sum()),
        'median_survival_days': seg_medians[seg],
        'logrank_p':  results_seg.p_value
    })

for ch, kmf in kmf_channels.items():
    sub = df[df['primary_channel'] == ch]
    survival_summary.append({
        'group_type': 'Channel',
        'group':      ch,
        'n':          len(sub),
        'events':     int(sub['churned'].sum()),
        'median_survival_days': ch_medians[ch],
        'logrank_p':  results_ch.p_value
    })

for reg, kmf in kmf_regions.items():
    sub = df[df['primary_region'] == reg]
    survival_summary.append({
        'group_type': 'Region',
        'group':      reg,
        'n':          len(sub),
        'events':     int(sub['churned'].sum()),
        'median_survival_days': reg_medians[reg],
        'logrank_p':  results_reg.p_value
    })

summary_df = pd.DataFrame(survival_summary)
output_path = processed_path / 'km_survival_summary.csv'
summary_df.to_csv(output_path, index=False)

print(f'Exported: km_survival_summary.csv ({len(summary_df)} rows)')
print(summary_df.to_string(index=False))